## Google Cloud Monitoring Essentials

# Google Cloud Monitoring Essentials

Monitoring your application is a massive step forward! Tracking metrics and building dashboards is like having a **mission control** for your code. Ready to launch? 🚀

---

## Introduction: The Importance of Application Monitoring

Welcome to the next step in your GCP developer journey! So far, you have learned how to keep your applications secure and protect your APIs. Now, we focus on a critical aspect: **Monitoring**.

Monitoring is essential for both security and reliability. It helps you answer:
*   Is my application working as expected?
*   Are there any unusual patterns or errors?
*   How is my application performing over time?

**Google Cloud Monitoring** (formerly Stackdriver) helps you collect, view, and analyze data. In this lesson, you will learn to send custom data (**metrics**) using **OpenTelemetry**, an industry-standard observability framework, and visualize them on dashboards.



---

## Quick Recall: Client Libraries vs. OpenTelemetry

In previous lessons, you used official GCP client libraries for Secret Manager or Cloud Functions. For monitoring, you *could* use the official library:

```python
from google.cloud import monitoring_v3
client = monitoring_v3.MetricServiceClient()
```

However, we will use **OpenTelemetry**. It is a vendor-neutral standard that makes your code more portable and provides a cleaner API.

### Benefits of OpenTelemetry:
*   **Vendor-neutral:** No "lock-in" to a specific cloud provider.
*   **Cleaner API:** Intuitive and simple code.
*   **Industry standard:** Widely adopted across the tech world.



---

## Understanding the Metric Flow

Let's visualize how the pieces fit together:

1.  **Your Application:** Uses the OpenTelemetry API to record metrics.
2.  **OpenTelemetry SDK:** Processes and exports these metrics.
3.  **Cloud Monitoring:** Receives and stores the data.
4.  **Dashboard:** Displays the metrics visually.

---

## Setting Up OpenTelemetry

### 1. Import Required Libraries
```python
import time
import random
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.resources import Resource
```

### 2. Configure the Provider
You need to identify your service and tell the SDK where to send the data (the **exporter**).

```python
# Identify your service
resource = Resource.create({"service.name": "OrdersAPI"})

# Configure the OTLP exporter (sending to a local collector in this environment)
exporter = OTLPMetricExporter(endpoint="http://localhost:4317", insecure=True)

# Export metrics every 10 seconds
reader = PeriodicExportingMetricReader(exporter, export_interval_millis=10000)

# Set the global meter provider
provider = MeterProvider(resource=resource, metric_readers=[reader])
metrics.set_meter_provider(provider)

# Create a meter to generate instruments
meter = metrics.get_meter("app.monitoring")
```

---

## Emitting Custom Metrics

We will use a **Histogram**, which is perfect for recording measurements like amounts, durations, or sizes.

### 3. Create a Histogram Instrument
```python
order_value_histogram = meter.create_histogram(
    name="order_value",
    description="Value of customer orders",
    unit="USD"
)
```

### 4. Record Values
Recording a value is straightforward. OpenTelemetry handles the timestamps and batching for you.

```python
for i in range(5):
    value = random.uniform(10, 200)
    order_value_histogram.record(value)
    print(f"Recorded metric: OrderValue = {value:.2f}")
    time.sleep(0.5)

# Ensure the last batch is exported before the script ends
time.sleep(10)
```

---

## Visualizing with Dashboards

A **Dashboard** is a visual display of your metrics. In a production GCP environment, you can create them via the console or via code using the `google-cloud-monitoring-dashboard` library.

### Dashboard Configuration Example
When creating a dashboard programmatically, you define **Widgets**. A widget for our histogram might look like this:

```python
v1.Widget(
    title="Order Value Distribution",
    xy_chart=v1.XyChart(
        data_sets=[
            v1.XyChart.DataSet(
                time_series_query=v1.TimeSeriesQuery(
                    time_series_filter=v1.TimeSeriesFilter(
                        # Note the prefix used by OTLP
                        filter='metric.type="workload.googleapis.com/order_value" AND resource.labels.service_name="OrdersAPI"',
                        aggregation=v1.Aggregation(
                            alignment_period={"seconds": 300},
                            per_series_aligner=v1.Aggregation.Aligner.ALIGN_MEAN,
                        ),
                    )
                )
            )
        ]
    )
)
```

> **Note on Resource Labels:** In real environments, filters change based on where your code runs (Cloud Run vs. GKE). You can use the **Metrics Explorer** in the Google Cloud Console to find the exact filter syntax for your specific environment.



---

## Summary And Practice Preview

In this lesson, you learned how to:
*   Use **OpenTelemetry** to emit custom metrics.
*   Configure **OTLP** exporters to send data to Cloud Monitoring.
*   Create **Histograms** to track numerical distributions.
*   Conceptualize **Dashboards** for real-time visualization.

Monitoring makes your applications more resilient and easier to manage. In the upcoming exercises, you will focus on writing the Python code to emit your own metrics!

## Create a Histogram and Record Values

You have been given a partially configured OpenTelemetry setup that already connects to the Cloud Monitoring backend. The resource, exporter, and meter are all configured correctly. However, two critical pieces are missing to actually emit metrics.

Your objective is to complete the monitoring setup by:

    Creating a histogram instrument to track order values
    Recording metric values using the histogram

Look for the #TODO comments in the code — they will guide you to exactly where you need to add the missing pieces. You'll need to use meter.create_histogram() to create the instrument with the name "order_value", description "Value of customer orders", and unit "USD". Then you'll use the record() method on the histogram to send each order value.

Once you complete the code, it will emit five sample order value metrics to Cloud Monitoring. This hands-on practice will help you master the essential skill of using OpenTelemetry to send custom metrics for monitoring your applications.
Suggestions

```python
import time
import random
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.resources import Resource

# Create a resource to identify your service
resource = Resource.create({"service.name": "OrdersAPI"})

# Create the OTLP exporter (for local development)
exporter = OTLPMetricExporter(endpoint="http://localhost:4317", insecure=True)

# Create a metric reader that exports every 10 seconds
reader = PeriodicExportingMetricReader(exporter, export_interval_millis=10000)

# Create and set the meter provider
provider = MeterProvider(resource=resource, metric_readers=[reader])
metrics.set_meter_provider(provider)

# Get a meter for creating instruments
meter = metrics.get_meter("app.monitoring")

#TODO: Create a histogram instrument named "order_value" with description "Value of customer orders" and unit "USD"
#TODO: Store it in a variable called order_value_histogram

# Record metric values
for i in range(5):
    value = random.uniform(10, 200)
    #TODO: Record the value using order_value_histogram.record()
    print(f"Recorded metric: OrderValue = {value:.2f}")
    time.sleep(0.5)

# Give the exporter time to send the final batch
time.sleep(10)
```

To complete your monitoring setup, you need to initialize the metric instrument and then call its recording method within your loop. This allows OpenTelemetry to capture the distribution of your order values and prepare them for export to Google Cloud Monitoring.

### Updated `main.py`

Here is the completed code with the histogram creation and recording logic:

```python
import time
import random
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.resources import Resource

# Create a resource to identify your service
resource = Resource.create({"service.name": "OrdersAPI"})

# Create the OTLP exporter (for local development)
exporter = OTLPMetricExporter(endpoint="http://localhost:4317", insecure=True)

# Create a metric reader that exports every 10 seconds
reader = PeriodicExportingMetricReader(exporter, export_interval_millis=10000)

# Create and set the meter provider
provider = MeterProvider(resource=resource, metric_readers=[reader])
metrics.set_meter_provider(provider)

# Get a meter for creating instruments
meter = metrics.get_meter("app.monitoring")

# FIX: Create a histogram instrument named "order_value"
order_value_histogram = meter.create_histogram(
    name="order_value",
    description="Value of customer orders",
    unit="USD"
)

# Record metric values
for i in range(5):
    value = random.uniform(10, 200)
    
    # FIX: Record the value using order_value_histogram.record()
    order_value_histogram.record(value)
    
    print(f"Recorded metric: OrderValue = {value:.2f}")
    time.sleep(0.5)

# Give the exporter time to send the final batch
time.sleep(10)
```

---

### Key Concepts Applied



*   **Histogram Creation**: By using `meter.create_histogram()`, you are defining a "bucketed" metric. Unlike a simple counter, a histogram allows Cloud Monitoring to calculate the mean, min, max, and percentiles (like the 95th percentile) of your order values.
*   **Recording**: The `.record(value)` method doesn't send the data to GCP immediately. Instead, the SDK aggregates these values in memory. 
*   **Exporting**: The `PeriodicExportingMetricReader` you configured will wake up every 10,000 milliseconds (10 seconds) to flush those aggregated values to the backend. This is why the final `time.sleep(10)` is necessary—it ensures the script doesn't exit before the background export thread finishes its work.

### Pro-Tip: Naming Conventions
In OpenTelemetry, metric names should ideally follow a namespace pattern (e.g., `orders.value` or `inventory.item_count`). Using clear names and units like `USD` makes it much easier for other developers to understand your charts in the Google Cloud Monitoring dashboard.

## Batch Emitting Multiple Metrics in a Single Call

Now that you know how to create a histogram instrument and record metrics, let's expand your monitoring by tracking multiple metrics from your application.

Suppose you want to track three different metrics from your application:

    order_value — the value of an order (in USD)
    order_count — the number of orders processed
    processing_time — how long it took to process the orders (in milliseconds)

Your task is to create histogram instruments for all three metrics. The starter code already includes the first histogram (order_value) as an example. You need to create two more histogram instruments by filling in the #TODO comments.

Each new histogram should follow the same pattern as the first one: use meter.create_histogram() with an appropriate name, description, and unit. This will help you practice creating multiple metric instruments with OpenTelemetry.

```python
import time
import random
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.resources import Resource

# Create a resource to identify your service
resource = Resource.create({"service.name": "OrdersAPI"})

# Create the OTLP exporter (for local development)
exporter = OTLPMetricExporter(endpoint="http://localhost:4317", insecure=True)

# Create a metric reader that exports every 10 seconds
reader = PeriodicExportingMetricReader(exporter, export_interval_millis=10000)

# Create and set the meter provider
provider = MeterProvider(resource=resource, metric_readers=[reader])
metrics.set_meter_provider(provider)

# Get a meter for creating instruments
meter = metrics.get_meter("app.monitoring")

# Create histograms for different metrics
order_value_histogram = meter.create_histogram(
    name="order_value",
    description="Value of customer orders",
    unit="USD"
)

# TODO: Create a histogram for order_count with name="order_count", 
# description="Number of orders processed", and unit="orders"

# TODO: Create a histogram for processing_time with name="processing_time",
# description="Time taken to process orders", and unit="ms"

# Record metric values
for i in range(5):
    order_value = random.uniform(10, 200)
    order_count = random.randint(1, 10)
    processing_time = random.uniform(50, 500)
    
    order_value_histogram.record(order_value)
    # TODO: Record order_count to order_count_histogram
    # TODO: Record processing_time to processing_time_histogram
    
    print(f"Recorded metrics: OrderValue={order_value:.2f}, OrderCount={order_count}, ProcessingTime={processing_time:.2f}ms")
    time.sleep(0.5)

# Give the exporter time to send the final batch
time.sleep(10)

```

Tracking multiple metrics simultaneously allows you to see the relationship between different parts of your system—for example, how a spike in `order_count` might correlate with an increase in `processing_time`.

By using OpenTelemetry, you can bundle these different "instruments" under the same service name, making them easy to filter and compare within Google Cloud Monitoring.

### Updated `main.py`

Here is the completed implementation for all three histograms:

```python
import time
import random
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.resources import Resource

# Create a resource to identify your service
resource = Resource.create({"service.name": "OrdersAPI"})

# Create the OTLP exporter (for local development)
exporter = OTLPMetricExporter(endpoint="http://localhost:4317", insecure=True)

# Create a metric reader that exports every 10 seconds
reader = PeriodicExportingMetricReader(exporter, export_interval_millis=10000)

# Create and set the meter provider
provider = MeterProvider(resource=resource, metric_readers=[reader])
metrics.set_meter_provider(provider)

# Get a meter for creating instruments
meter = metrics.get_meter("app.monitoring")

# Create histograms for different metrics
order_value_histogram = meter.create_histogram(
    name="order_value",
    description="Value of customer orders",
    unit="USD"
)

# FIX: Created histogram for order_count
order_count_histogram = meter.create_histogram(
    name="order_count", 
    description="Number of orders processed", 
    unit="orders"
)

# FIX: Created histogram for processing_time
processing_time_histogram = meter.create_histogram(
    name="processing_time",
    description="Time taken to process orders", 
    unit="ms"
)

# Record metric values
for i in range(5):
    order_value = random.uniform(10, 200)
    order_count = random.randint(1, 10)
    processing_time = random.uniform(50, 500)
    
    # Record all three metrics
    order_value_histogram.record(order_value)
    # FIX: Recorded order_count
    order_count_histogram.record(order_count)
    # FIX: Recorded processing_time
    processing_time_histogram.record(processing_time)
    
    print(f"Recorded metrics: OrderValue={order_value:.2f}, OrderCount={order_count}, ProcessingTime={processing_time:.2f}ms")
    time.sleep(0.5)

# Give the exporter time to send the final batch
time.sleep(10)
```

---

### Why Use Three Different Histograms?



Using separate histograms for these metrics provides distinct insights:

1.  **Order Value (USD):** Tracks the financial health and sales distribution.
2.  **Order Count (orders):** Helps you understand throughput—how busy is your system at any given moment?
3.  **Processing Time (ms):** A classic **latency** metric. This is vital for maintaining a good user experience. If this number creeps up while `order_count` stays the same, you may have a code performance issue.

### The "Z" in the Time
Notice that while we record these metrics locally, the `PeriodicExportingMetricReader` handles the heavy lifting of organizing this data into **Time Series**. In Google Cloud Monitoring, each of these histograms will appear as a separate `metric.type` under the `[workload.googleapis.com/](https://workload.googleapis.com/)` prefix, allowing you to build complex multi-chart dashboards in seconds.

## Fix OTel Configuration and Export Timing

Perfect! You've learned how to emit metrics using OpenTelemetry. Now, let's address two common issues that can prevent your metrics from being properly exported to Cloud Monitoring.

You have a monitoring setup that creates and records metrics, but the metrics aren't being exported because of two missing pieces in the configuration.

Your task is to fix two specific issues in the code:

    Add the missing line that registers the meter provider with OpenTelemetry, so the system knows to use your configured exporter.
    Add the necessary wait time at the end of the script to ensure all metrics are exported before the program exits.

Look for the #TODO comments in the code to find where these fixes are needed. Once you make these changes, your metrics will be properly exported through OTLP and your monitoring solution will work correctly.

```python
import time
import random
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.resources import Resource

# Create a resource to identify your service
resource = Resource.create({"service.name": "OrdersAPI"})

# Create the OTLP exporter (for local development)
exporter = OTLPMetricExporter(endpoint="http://localhost:4317", insecure=True)

# Create a metric reader that exports every 10 seconds
reader = PeriodicExportingMetricReader(exporter, export_interval_millis=10000)

# Create and set the meter provider
provider = MeterProvider(resource=resource, metric_readers=[reader])
# TODO: Add the line to set the meter provider globally so OpenTelemetry uses your configured exporter

# Get a meter for creating instruments
meter = metrics.get_meter("app.monitoring")

# Create a histogram for order values
order_value_histogram = meter.create_histogram(
    name="order_value",
    description="Value of customer orders",
    unit="USD"
)

# Record metric values
for i in range(5):
    value = random.uniform(10, 200)
    order_value_histogram.record(value)
    print(f"Recorded metric: OrderValue = {value:.2f}")
    time.sleep(0.5)

# TODO: Add a sleep to give the exporter time to send the final batch (10 seconds)
```

These two fixes are crucial because they bridge the gap between "collecting data" and "sending data." Without registering the provider, OpenTelemetry doesn't know where to route your recordings. Without the wait time, the script terminates while the background export thread is still packing the data.

### Updated `main.py`

Here is the corrected configuration to ensure your metrics reach Cloud Monitoring:

```python
import time
import random
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.resources import Resource

# Create a resource to identify your service
resource = Resource.create({"service.name": "OrdersAPI"})

# Create the OTLP exporter (for local development)
exporter = OTLPMetricExporter(endpoint="http://localhost:4317", insecure=True)

# Create a metric reader that exports every 10 seconds
reader = PeriodicExportingMetricReader(exporter, export_interval_millis=10000)

# Create and set the meter provider
provider = MeterProvider(resource=resource, metric_readers=[reader])

# FIX: Set the meter provider globally so metrics.get_meter() uses your config
metrics.set_meter_provider(provider)

# Get a meter for creating instruments
meter = metrics.get_meter("app.monitoring")

# Create a histogram for order values
order_value_histogram = meter.create_histogram(
    name="order_value",
    description="Value of customer orders",
    unit="USD"
)

# Record metric values
for i in range(5):
    value = random.uniform(10, 200)
    order_value_histogram.record(value)
    print(f"Recorded metric: OrderValue = {value:.2f}")
    time.sleep(0.5)

# FIX: Wait 10 seconds to ensure the PeriodicExportingMetricReader flushes the final batch
time.sleep(10)
```

---

### Why These Steps are Critical



1.  **The Global Provider**: OpenTelemetry uses a "Pluggable" architecture. By calling `metrics.set_meter_provider(provider)`, you are telling the entire Python process: *"Whenever someone asks for a meter, give them one that is hooked up to my OTLP exporter."* Without this, the system defaults to a "No-Op" provider that simply ignores all data.
2.  **The Export Window**: The `PeriodicExportingMetricReader` runs on a separate background thread. It collects data in the background and sends it every 10 seconds. If your main script finishes its loop and closes immediately, that background thread is killed before it can finish the HTTP request to the collector.
3.  **Local vs. Production**: In a long-running production service (like a web server or a Cloud Function), you usually don't need the final `time.sleep(10)` because the process stays alive. However, for scripts, CLI tools, or batch jobs, this "graceful shutdown" period is mandatory to prevent data loss.

## Build a Complete Application Monitoring Workflow in Python

You are now ready to set up application monitoring using OpenTelemetry! Your task is to implement a complete monitoring workflow that tracks order values from your application.

Here's what you need to do:

    Configure the OpenTelemetry resource to identify your service.
    Set up the OTLP exporter to send metrics to the monitoring system.
    Create and configure the meter provider with a periodic metric reader.
    Create a histogram instrument to track order values.
    Record sample metric values in a loop.

Look for the #TODO comments in the code — they will guide you to the exact places where you need to add code. Make sure to follow the setup sequence carefully, as each component depends on the previous one.

When you're done, your application will emit custom metrics to Cloud Monitoring using the industry-standard OpenTelemetry framework!
****

```python
import time
import random
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.resources import Resource

# TODO: Create a resource with the service.name attribute set to "OrdersAPI"
# Hint: Use Resource.create() with a dictionary containing {"service.name": "OrdersAPI"}

# TODO: Create an OTLP exporter pointing to "http://localhost:4317" with insecure=True
# Hint: Use OTLPMetricExporter(endpoint=..., insecure=...)

# TODO: Create a PeriodicExportingMetricReader that exports every 10 seconds (10000 milliseconds)
# Hint: Pass the exporter and export_interval_millis to PeriodicExportingMetricReader

# TODO: Create a MeterProvider with the resource and metric_readers (as a list containing the reader)
# Hint: MeterProvider(resource=..., metric_readers=[...])

# TODO: Set the global meter provider using metrics.set_meter_provider()

# TODO: Get a meter named "app.monitoring" using metrics.get_meter()

# TODO: Create a histogram instrument with:
# - name: "order_value"
# - description: "Value of customer orders"
# - unit: "USD"
# Hint: Use meter.create_histogram()

# TODO: Create a for loop that runs 5 times
    # TODO: Generate a random value between 10 and 200 using random.uniform(10, 200)
    # TODO: Record the value to the histogram using order_value_histogram.record()
    # TODO: Print the recorded value in the format: "Recorded metric: OrderValue = {value:.2f}"
    # TODO: Sleep for 0.5 seconds using time.sleep(0.5)

# TODO: Sleep for 10 seconds to allow the exporter to send the final batch
# Hint: Use time.sleep(10)
```